# DEEZER API PULL

11/5/26 script to pull deezer data from their public API.
- have genre
- can pull top 50 artists in each genre (url format: base + /genre + /genre_id + /artists)
- goal: dataframe of top 50 artists per genre

functions:
- get genre
- get artists from genre
- validation
- make dataframe

ok so the top 50 doesnt work well, what we're gonna do is get the top x tracks across all genres, plays, and get the top artist by plays of each days, top x in most genres, yesterdays, past week, past month (when available). lets also make this just deezer

In [ ]:
import argparse
import requests
import pandas as pd

import io
import logging
import time
from datetime import datetime, timezone

import boto3
from botocore.exceptions import ClientError

In [2]:
MAX_RETRIES = 3 # still need to implement this
url = 'https://api.deezer.com'
session = requests.Session()

In [3]:
# error class so we know where the error comes from further down the line/when the script is run
class DeezerError(Exception):
    pass

# DEFINE FUNCTIONS

In [4]:
def get_genre_data() -> pd.DataFrame:
    try:
        response = session.get(f'{url}/genre', timeout=15)
        response.raise_for_status()
        body = response.json() # the whole response
        if "error" in body:
            raise DeezerError('error in genres data body')
        df = pd.DataFrame(body["data"])[['id', 'name']]
        return df[df['id'] != 0] # no 'all' genre
    except requests.RequestException as e:
        print("request failed:", e)
        raise
    except DeezerError as e:
        print("deezer error:", e)
        raise

genres = get_genre_data()

In [5]:
def get_top_tracks() -> pd.DataFrame:
    try:
        result = []
        for genre_id, genre in zip(genres['id'], genres['name']):
            response = session.get(
                f'{url}/chart/{genre_id}/tracks',
                params={'limit': 100},
                timeout=15,)
            response.raise_for_status()
            body = response.json() # the whole response
            if "error" in body:
                raise DeezerError('error in artists data body')
            # dataframe transformation
            df = pd.DataFrame(body['data'])
            df['Genre'] = genre # add in genre data
            result.append(df)
            # print(len(result))
            # time.sleep(.5)
        return pd.concat(result, ignore_index=True)
    except requests.RequestException as e:
            print("request failed:", e)
            raise
    except DeezerError as e:
            print("deezer error:", e)
            raise

top_tracks = get_top_tracks()

In [6]:
# expanding out the artist column from a dictionary to separate columns to be added back to the tracks dataframe
artist_info_df = pd.json_normalize(top_tracks['artist'])
artist_info_df = artist_info_df.rename(columns={'id':'artist_id',
                                                'name':'artist_name',
                                                'link':'artist_link',
                                                'picture':'artist_picture',
                                                'tracklist':'top_tracklist'}, errors='ignore')[['artist_id','artist_name','artist_link','artist_picture','top_tracklist']]

In [7]:
album_info_df = pd.json_normalize(top_tracks['album'])
album_info_df = album_info_df.rename(columns={'id':'album_id',
                                              'title':'album_title',
                                              'cover':'album_cover',
                                              'tracklist':'album_tracklist'}, errors='ignore')[['album_id','album_title','album_cover','album_tracklist']]

In [8]:
top_tracks_expanded = pd.concat([top_tracks, artist_info_df, album_info_df], axis=1).drop(columns=['artist', 'album'], errors='ignore')
top_tracks_expanded.sample(1)

,id,title,title_short,title_version,link,duration,rank,explicit_lyrics,explicit_content_lyrics,explicit_content_cover,...,Genre,artist_id,artist_name,artist_link,artist_picture,top_tracklist,album_id,album_title,album_cover,album_tracklist
894,101391228,Casanova 70 (The Secret of Cool),Casanova 70,(The Secret of Cool),https://www.deezer.com/track/101391228,331,238305,False,0,0,...,Electro,369,Air,https://www.deezer.com/artist/369,https://api.deezer.com/artist/369/image,https://api.deezer.com/artist/369/top?limit=50,10526066,"The Remixes, Vol. 1",https://api.deezer.com/album/10526066/image,https://api.deezer.com/album/10526066/tracks


In [9]:
# deduplicated artist list
artist_list = top_tracks_expanded['artist_id'].unique().tolist()
len(artist_list)

1660

In [14]:
import random

In [15]:
# changed, rewrite this to get the artists from top tracks and get their number of fans
def get_artist_data() -> pd.DataFrame:
    try:
        result = []
        for artist_id in random.sample(artist_list, 100): # just testing with one artist for now
            response = session.get(f'{url}/artist/{artist_id}', timeout=15)
            response.raise_for_status()
            body = response.json() # the whole response
            if "error" in body:
                raise DeezerError('error in artists data body')
            result.append(body)
        return pd.DataFrame(result)
    except requests.RequestException as e:
            print("request failed:", e)
            raise
    except DeezerError as e:
            print("deezer error:", e)
            raise

artists_data = get_artist_data()

# DATA VALIDATION

add checks for nulls, missing genres, artist names, etc

to validate:
- genres
    - not much to do here. check for null data
- top_tracks
    - handle nulls, less than 100 tracks in genre
- artists_df
    - handle null number of fans, missing data in join

# UPLOAD TO S3

In [ ]:
s3 = boto3.client('s3')

def upload_df(df: pd.DataFrame, bucket: str, key: str) -> bool:
    """Write a DataFrame to S3 as Parquet."""
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False)   # needs pyarrow
    buffer.seek(0)
    try:
        s3.upload_fileobj(buffer, bucket, key)
    except ClientError as e:
        logging.error(e)
        return False
    return True

In [20]:
upload_df(genres, 'deezer-api-practice', f'genres_{datetime.now(timezone.utc).strftime("%Y-%m-%d")}.parquet')
upload_df(top_tracks_expanded, 'deezer-api-practice', f'top_tracks_{datetime.now(timezone.utc).strftime("%Y-%m-%d")}.parquet')
upload_df(artists_data, 'deezer-api-practice', f'artists_{datetime.now(timezone.utc).strftime("%Y-%m-%d")}.parquet')

True

# add all this to main so that it's a python script and package